### Currency convertor Agent 

In [152]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model = 'ministral-3:3b')

In [153]:
from langchain_core.tools import tool
import requests

@tool 
def currency_convertor(amount: int, from_currency: str, to: str) -> float:
    '''convert currency by using ISO 4217 Three Letter Currency Codes - e.g. USD for US Dollars, EUR for Euros, JPY for Japanese Yen etc'''

    # result = requests.get(f"https://v6.exchangerate-api.com/v6/{'a3a9b4d36225c6a18ae6ff54'}/latest/{from_currency}")
    result = requests.get(f"https://v6.exchangerate-api.com/v6/{'a3a9b4d36225c6a18ae6ff54'}/pair/{from_currency}/{to}")

    print(result.json()['conversion_rate'] * amount)
    print(result.json())

    conversion_result = result.json()['conversion_rate'] * amount

    return f"{conversion_result} will be the value of {amount} {from_currency} to {to}"

    

In [154]:
currency_convertor.invoke({"amount": 1, "from_currency": "USD", "to": "INR"})

90.9589
{'result': 'success', 'documentation': 'https://www.exchangerate-api.com/docs', 'terms_of_use': 'https://www.exchangerate-api.com/terms', 'time_last_update_unix': 1768867201, 'time_last_update_utc': 'Tue, 20 Jan 2026 00:00:01 +0000', 'time_next_update_unix': 1768953601, 'time_next_update_utc': 'Wed, 21 Jan 2026 00:00:01 +0000', 'base_code': 'USD', 'target_code': 'INR', 'conversion_rate': 90.9589}


'90.9589 will be the value of 1 USD to INR'

In [155]:
llm_with_tool = llm.bind_tools([currency_convertor])

In [176]:
from langchain_core.messages import HumanMessage


messages = []

query = HumanMessage(content="hi, can you convert 100 indian rupees to european British Pound currency for me?")

messages.append(query)


In [177]:
# Tool Calling LLM 

tool_calling = llm_with_tool.invoke(messages)

In [178]:
tool_calling

AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'ministral-3:3b', 'created_at': '2026-01-20T21:19:26.4615893Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2342807500, 'load_duration': 324147800, 'prompt_eval_count': 674, 'prompt_eval_duration': 805684500, 'eval_count': 29, 'eval_duration': 1186631400, 'logprobs': None, 'model_name': 'ministral-3:3b', 'model_provider': 'ollama'}, id='lc_run--019bdd46-cfd3-7eb0-8877-bc6b3208a344-0', tool_calls=[{'name': 'currency_convertor', 'args': {'amount': 100, 'from_currency': 'INR', 'to': 'GBP'}, 'id': 'a187b72e-5298-44ec-846f-f5649d8bccf6', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 674, 'output_tokens': 29, 'total_tokens': 703})

In [179]:
messages.append(tool_calling)

In [180]:
tool_calling.tool_calls[0]

{'name': 'currency_convertor',
 'args': {'amount': 100, 'from_currency': 'INR', 'to': 'GBP'},
 'id': 'a187b72e-5298-44ec-846f-f5649d8bccf6',
 'type': 'tool_call'}

In [181]:
tool_execution_result  = currency_convertor.invoke(tool_calling.tool_calls[0])

0.8201
{'result': 'success', 'documentation': 'https://www.exchangerate-api.com/docs', 'terms_of_use': 'https://www.exchangerate-api.com/terms', 'time_last_update_unix': 1768867201, 'time_last_update_utc': 'Tue, 20 Jan 2026 00:00:01 +0000', 'time_next_update_unix': 1768953601, 'time_next_update_utc': 'Wed, 21 Jan 2026 00:00:01 +0000', 'base_code': 'INR', 'target_code': 'GBP', 'conversion_rate': 0.008201}


In [182]:
tool_execution_result

ToolMessage(content='0.8201 will be the value of 100 INR to GBP', name='currency_convertor', tool_call_id='a187b72e-5298-44ec-846f-f5649d8bccf6')

In [183]:
messages.append(tool_execution_result)

In [184]:
# now finally our messages list looks like this 
messages

[HumanMessage(content='hi, can you convert 100 indian rupees to european British Pound currency for me?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'ministral-3:3b', 'created_at': '2026-01-20T21:19:26.4615893Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2342807500, 'load_duration': 324147800, 'prompt_eval_count': 674, 'prompt_eval_duration': 805684500, 'eval_count': 29, 'eval_duration': 1186631400, 'logprobs': None, 'model_name': 'ministral-3:3b', 'model_provider': 'ollama'}, id='lc_run--019bdd46-cfd3-7eb0-8877-bc6b3208a344-0', tool_calls=[{'name': 'currency_convertor', 'args': {'amount': 100, 'from_currency': 'INR', 'to': 'GBP'}, 'id': 'a187b72e-5298-44ec-846f-f5649d8bccf6', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 674, 'output_tokens': 29, 'total_tokens': 703}),
 ToolMessage(content='0.8201 will be the value of 100 INR to GBP', name='currency_convertor', tool_

In [185]:
from langchain_core.messages import SystemMessage

# adding the system message at the start of the messages list to ensure that the LLM uses the tool result exactly as provided

messages.insert(
    0,
    SystemMessage(
        content=(
            "You MUST use the tool result exactly as provided. "
            "Do NOT modify, estimate, or override tool outputs. just beautify the response for the user."
        )
    )
)

In [186]:
final_response = llm_with_tool.invoke(messages)

In [187]:
final_response

AIMessage(content='100 Indian Rupees (INR) is approximately **£0.82** in British Pounds (GBP).', additional_kwargs={}, response_metadata={'model': 'ministral-3:3b', 'created_at': '2026-01-20T21:19:49.6373158Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3228480400, 'load_duration': 304840900, 'prompt_eval_count': 204, 'prompt_eval_duration': 1907250200, 'eval_count': 29, 'eval_duration': 981579000, 'logprobs': None, 'model_name': 'ministral-3:3b', 'model_provider': 'ollama'}, id='lc_run--019bdd47-26e7-70b2-aa38-3c9da10241ad-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 204, 'output_tokens': 29, 'total_tokens': 233})